# 🛡️ SENTINEL — Phase 2 GRPO on Kaggle (FREE)

**OpenEnv Hackathon India 2026** · continues from the 200-step model already on HF Hub.

Strategy:
1. Pull `srikrish2004/sentinel-qwen3-4b-grpo` (the 200-step LoRA) from HF Hub.
2. Continue GRPO for another 150 steps with `temperature=0.7` and `max_new_tokens=1024` (the two key fixes).
3. Push the polished LoRA back to `srikrish2004/sentinel-qwen3-4b-grpo-phase2`.

**Hardware:** Kaggle GPU T4 ×2 (16 + 16 GB VRAM) — free.  
**Quota:** 30 GPU-hours/week, 12-hour session cap, runs after browser close.  
**ETA:** ~6-8 hours for 150 steps.  
**Cost:** $0.

---

## Before you run

1. Open this notebook on https://www.kaggle.com/code (New Notebook → upload this `.ipynb`).
2. **Settings → Accelerator → GPU T4 × 2** (or P100 if T4×2 unavailable).
3. **Settings → Persistence → Files only** (so `/kaggle/working` survives session restart).
4. Add your secrets in **Add-ons → Secrets**:
   - `HF_TOKEN` — write-scope HuggingFace token (revoke the one you exposed earlier and make a new one).
5. Click **Save Version → Save & Run All (Commit)** to run unattended for 9 hours after browser close.

## 1 · Detect environment + GPU sanity check

In [ ]:
import os, sys, subprocess, json
from pathlib import Path

IS_KAGGLE = Path('/kaggle').exists()
WORK_DIR  = Path('/kaggle/working') if IS_KAGGLE else Path('/content')
REPO_DIR  = WORK_DIR / 'openEnv'
OUT_DIR   = WORK_DIR / 'sentinel_phase2'
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'monitoring').mkdir(parents=True, exist_ok=True)

print(f'IS_KAGGLE = {IS_KAGGLE}')
print(f'WORK_DIR  = {WORK_DIR}')
print(f'REPO_DIR  = {REPO_DIR}')
print(f'OUT_DIR   = {OUT_DIR}')

subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], check=False)

## 2 · Clone repo + install deps (5-8 min)

In [ ]:
%cd {WORK_DIR}
if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/sri11223/openEnv.git
%cd {REPO_DIR}
!git pull --quiet || true

!pip install -q --upgrade pip
!pip install -q -r requirements-train.txt
!pip install -q unsloth peft trl bitsandbytes accelerate datasets huggingface_hub
print('✅ deps installed')

## 3 · HF login (uses Kaggle Secret `HF_TOKEN`)

In [ ]:
HF_TOKEN = ''
if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as exc:
        print(f'⚠️  Could not read Kaggle Secret HF_TOKEN: {exc}')
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

assert HF_TOKEN, 'Add HF_TOKEN under Add-ons → Secrets, then re-run this cell.'
os.environ['HF_TOKEN'] = HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print('✅ HF login ok')

## 4 · Pull the 200-step LoRA from HF Hub into the warm-start path

In [ ]:
from huggingface_hub import snapshot_download

WARM_START = WORK_DIR / 'warm_start_phase2' / 'final'
WARM_START.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id   = 'srikrish2004/sentinel-qwen3-4b-grpo',
    local_dir = str(WARM_START),
    local_dir_use_symlinks = False,
    token     = HF_TOKEN,
)
!ls -lh {WARM_START}
assert (WARM_START / 'adapter_model.safetensors').exists(), 'LoRA missing — check repo id.'
print('✅ Phase-1 LoRA staged at', WARM_START)

## 5 · Phase 2 config — the 2 changes that matter

| Knob | Phase 1 (done) | Phase 2 (now) | Why |
|---|---|---|---|
| `MAX_NEW_TOKENS` | 512 | **1024** | Phase 1 finished concise; Phase 2 needs room for `temp=0.7` reasoning |
| `GEN_TEMPERATURE` | 1.0 | **0.7** | Lower temp ⇒ less random over-flagging on `multi_crisis_command` |
| `TRAIN_STEPS` | 200 | 150 | Fits in one Kaggle session (12h cap) |
| `WARM_START_STEPS` | 0 | **1** | Triggers the SKIP-and-LOAD branch in `train.py` |
| `WARM_START_OUTPUT_DIR` | — | `/kaggle/working/warm_start_phase2` | Where we placed the Phase-1 LoRA |

In [ ]:
phase2_env = {
    'MODEL_NAME':                'unsloth/Qwen3-4B-bnb-4bit',
    'TRAIN_STEPS':               '150',
    'NUM_GENERATIONS':           '4',
    'MODEL_STEPS_LIMIT':         '3',
    'MAX_NEW_TOKENS':            '1024',
    'GEN_TEMPERATURE':           '0.7',
    'GEN_TOP_P':                 '0.95',
    'WARM_START_STEPS':          '1',
    'WARM_START_OUTPUT_DIR':     str(WORK_DIR / 'warm_start_phase2'),
    'USE_SENTINEL':              '1',
    'USE_CURRICULUM':            '1',
    'CURRICULUM_OPEN_WINDOW':    '1',
    'REWARD_SCHEDULE_MODE':      'dynamic',
    'ROLLOUT_AUDIT_EVERY':       '5',
    'ROLLOUT_AUDIT_SAMPLES':     '3',
    'OUTPUT_DIR':                str(OUT_DIR),
    'TRAIN_MONITOR_DIR':         str(OUT_DIR / 'monitoring'),
    'PYTORCH_ALLOC_CONF':        'expandable_segments:True',
    'TOKENIZERS_PARALLELISM':    'false',
    'HF_TOKEN':                  HF_TOKEN,
}
for k, v in phase2_env.items():
    os.environ[k] = v

print('Phase 2 environment locked in.')

## 6 · Launch Phase 2 GRPO (logs stream live, also saved to file)

When you click **Save & Run All (Commit)**, Kaggle will run this cell unattended for up to 12 hours. The notebook keeps running even when you close the browser.

What to watch:
- `Warm-start checkpoint found ...` ⇒ Phase-1 LoRA loaded ✓
- `clipped_ratio < 0.5` ⇒ 1024 tokens is enough ✓
- `reward` rising past `0.30` ⇒ Phase 2 is working ✓

## ▶️ Resume from checkpoint (fill this in if restarting)

If Phase 2 was interrupted, find your last checkpoint in the **Output** tab of
the previous run under `outputs/checkpoints/checkpoint-NNN/`.

Set `RESUME_FROM_CKPT` below to that path, then run this cell **before** the training cell.
Leave it as `''` for a fresh start.

In [ ]:
# ── RESUME CONFIG ────────────────────────────────────────────────────────────
# Set to e.g. "/kaggle/working/outputs/checkpoints/checkpoint-25" if resuming.
# Leave as "" for a fresh Phase 2 run.
RESUME_FROM_CKPT = ""  # <-- fill in checkpoint path here if resuming

if RESUME_FROM_CKPT:
    import os
    os.environ["RESUME_FROM"] = RESUME_FROM_CKPT
    print(f"\u2705 Will resume from: {RESUME_FROM_CKPT}")
else:
    os.environ.pop("RESUME_FROM", None)
    print("\u2139\ufe0f  Fresh run (no checkpoint resume)")


In [ ]:
%cd {REPO_DIR}
log_path = OUT_DIR / 'train_run_phase2.log'
print(f'📝 streaming logs -> {log_path}')
!python train.py 2>&1 | tee {log_path}

## 7 · Verify the polished LoRA was saved

In [ ]:
final_dir = OUT_DIR / 'final'
!ls -lh {final_dir}
assert (final_dir / 'adapter_model.safetensors').exists(), 'No final adapter — check the log above for errors.'
print('✅ Phase 2 LoRA at', final_dir)

## 8 · Push polished LoRA to HF Hub (separate repo so Phase 1 stays intact)

In [ ]:
from huggingface_hub import HfApi, create_repo

PHASE2_REPO = 'srikrish2004/sentinel-qwen3-4b-grpo-phase2'
create_repo(PHASE2_REPO, token=HF_TOKEN, exist_ok=True, private=False)
HfApi().upload_folder(
    folder_path    = str(final_dir),
    repo_id        = PHASE2_REPO,
    token          = HF_TOKEN,
    commit_message = 'Phase 2 GRPO: 150 more steps from sentinel-qwen3-4b-grpo at temp=0.7, max_tokens=1024',
)
print(f'✅ uploaded -> https://huggingface.co/{PHASE2_REPO}')

## 9 · Quick reward summary (for the README / model card)

In [ ]:
import re, statistics
log = (OUT_DIR / 'train_run_phase2.log').read_text(errors='ignore')
rewards = [float(m.group(1)) for m in re.finditer(r"'reward':\s*'([\d.]+)'", log)]
if rewards:
    print(f'steps logged       : {len(rewards)}')
    print(f'first-10 mean      : {statistics.mean(rewards[:10]):.3f}')
    print(f'last-10  mean      : {statistics.mean(rewards[-10:]):.3f}')
    print(f'overall max        : {max(rewards):.3f}')
else:
    print('No reward lines found yet — training may not have started.')

## What to do after this notebook finishes

1. Note the **first-10 vs last-10 reward** numbers above.
2. Update `hf_model_card.md` with the new last-10 mean (target: 0.45+).
3. Update `README.md` Phase 2 row.
4. The polished LoRA lives at `https://huggingface.co/srikrish2004/sentinel-qwen3-4b-grpo-phase2` — anyone can pull it with `PeftModel.from_pretrained(...)`.

If reward DROPS below the Phase-1 baseline of 0.30 (it shouldn't, but if it does):
- The `temp=0.7` was probably too aggressive — bump back to 0.85.
- Or the warm-start LoRA didn't load — re-check cell 4 + log for `Loaded warm-start LoRA from ...`.